# P2 ETF Siamese Ranker — Data Exploration
EDA notebook for the source dataset and feature engineering pipeline.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

## 1. Load Source Data

In [ ]:
from dataset import load_source_data
df = load_source_data()
print(f'Shape: {df.shape}')
print(f'Date range: {df.index.min().date()} → {df.index.max().date()}')
df.tail(3)

## 2. Universe Coverage (NaN audit)

In [ ]:
FI_UNIVERSE     = ['TLT','LQD','HYG','VNQ','GLD','SLV','VCIT']
EQUITY_UNIVERSE = ['QQQ','XLK','XLF','XLE','XLV','XLI','XLY','XLP','XLU','XME','GDX','IWM','XLB','XLRE']
BENCHMARKS      = ['SPY', 'AGG']
MACRO           = ['VIX','DXY','T10Y2Y','TBILL_3M','IG_SPREAD','HY_SPREAD']

all_cols = FI_UNIVERSE + EQUITY_UNIVERSE + BENCHMARKS + MACRO
coverage = pd.DataFrame({
    'first_valid': df[all_cols].apply(lambda s: s.first_valid_index()),
    'last_valid':  df[all_cols].apply(lambda s: s.last_valid_index()),
    'pct_complete': (df[all_cols].notna().sum() / len(df) * 100).round(1),
})
coverage

## 3. Price Normalised — FI Module

In [ ]:
fig, ax = plt.subplots()
for etf in FI_UNIVERSE:
    s = df[etf].dropna()
    (s / s.iloc[0]).plot(ax=ax, label=etf, linewidth=1.2)
ax.axhline(1, color='grey', linestyle='--', linewidth=0.8)
ax.legend(ncol=4, fontsize=9)
ax.set_title('FI / Commodities — Normalised Prices')
plt.tight_layout()
plt.show()

## 4. Price Normalised — Equity Module

In [ ]:
fig, ax = plt.subplots()
for etf in EQUITY_UNIVERSE:
    s = df[etf].dropna()
    (s / s.iloc[0]).plot(ax=ax, label=etf, linewidth=1.2)
ax.axhline(1, color='grey', linestyle='--', linewidth=0.8)
ax.legend(ncol=4, fontsize=9)
ax.set_title('Equity Sectors — Normalised Prices')
plt.tight_layout()
plt.show()

## 5. Macro Features Over Time

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 7))
for ax, col in zip(axes.flatten(), MACRO):
    df[col].dropna().plot(ax=ax, color='#6c63ff', linewidth=1)
    ax.set_title(col, fontsize=10)
    ax.set_xlabel('')
plt.suptitle('Macro Features', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 6. Feature Engineering — Sample

In [ ]:
from dataset import build_feature_matrix
feat_dict = build_feature_matrix(df, FI_UNIVERSE)
print('Feature columns:', feat_dict['GLD'].columns.tolist())
feat_dict['GLD'].dropna().tail(3)

## 7. Pairwise Dataset — Sample

In [ ]:
from dataset import build_pairwise_dataset, temporal_split

# Use only 2 years for speed in notebook
df_2y = df[df.index >= '2023-01-01']

Xi, Xj, y, dates = build_pairwise_dataset(df_2y, FI_UNIVERSE[:3], horizon=1)
print(f'Pairs: {len(y):,}  |  Xi shape: {Xi.shape}  |  Label balance: {y.mean():.3f}')

splits = temporal_split(Xi, Xj, y, dates)
print(f"Train: {len(splits['train'][2]):,}  Val: {len(splits['val'][2]):,}  Test: {len(splits['test'][2]):,}")

## 8. Rolling Correlation Heat Map (FI)

In [ ]:
import seaborn as sns
ret = df[FI_UNIVERSE].pct_change().dropna()
corr = ret.corr()
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('FI Universe — Return Correlation (Full Period)')
plt.tight_layout()
plt.show()